# 02 Cleaning & Transformation

**Purpose:** Clean the raw retail dataset by handling missing values, standardizing formats, and engineering new features. Each step is documented with its business justification.

In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW_DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'retail_store_sales.csv'
PROCESSED_DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_dataset.csv'

df = pd.read_csv(RAW_DATA_PATH)
print(f"Initial shape: {df.shape}")

Initial shape: (12575, 11)


### Step 1: Standardize Column Names
**Why:** Spaces and mixed casing make column referencing prone to errors. We will convert all columns to `snake_case`.

In [2]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
print("New columns:", df.columns.tolist())

New columns: ['transaction_id', 'customer_id', 'category', 'item', 'price_per_unit', 'quantity', 'total_spent', 'payment_method', 'location', 'transaction_date', 'discount_applied']


### Step 2: Identify and Handle Duplicates
**Why:** Exact duplicate rows can artificially inflate revenue and transaction counts.

In [3]:
initial_len = len(df)
df = df.drop_duplicates()
print(f"Dropped {initial_len - len(df)} duplicate rows.")

Dropped 0 duplicate rows.


### Step 3: Handle Missing Values
**Why:** Null values break aggregate functions and machine learning models. 
- For `price_per_unit`, we will impute with the median price to preserve data volume without skewing the distribution.
- For `total_spent`, we will attempt to recover it by multiplying `price_per_unit` * `quantity`. If it cannot be recovered, the row will be dropped as it represents invalid transaction data.

In [4]:
print("Nulls before handling:\n", df.isnull().sum())

# Impute missing price_per_unit with median
if df['price_per_unit'].isnull().any():
    median_price = df['price_per_unit'].median()
    df['price_per_unit'] = df['price_per_unit'].fillna(median_price)
    print(f"\nImputed missing price_per_unit with median: {median_price}")

# Recover total_spent where possible
if df['total_spent'].isnull().any():
    df['total_spent'] = df['total_spent'].fillna(df['price_per_unit'] * df['quantity'])

# Drop remaining rows with null total_spent or quantity
df = df.dropna(subset=['total_spent', 'quantity'])
print("\nNulls after handling:\n", df.isnull().sum())

Nulls before handling:
 transaction_id         0
customer_id            0
category               0
item                1213
price_per_unit       609
quantity             604
total_spent          604
payment_method         0
location               0
transaction_date       0
discount_applied    4199
dtype: int64

Imputed missing price_per_unit with median: 23.0

Nulls after handling:
 transaction_id         0
customer_id            0
category               0
item                 609
price_per_unit         0
quantity               0
total_spent            0
payment_method         0
location               0
transaction_date       0
discount_applied    3988
dtype: int64


### Step 4: Fix Category Variations
**Why:** Inconsistent casing (e.g., 'electronics' vs 'Electronics') splits a single category into multiple groups. We will standardize all string categories to Title Case.

In [5]:
cat_cols = ['category', 'item', 'payment_method', 'location']
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].str.title()
        
print("Unique categories:", df['category'].unique())

Unique categories: <StringArray>
[                        'Patisserie',                      'Milk Products',
                           'Butchers',                          'Beverages',
                               'Food',                          'Furniture',
      'Electric Household Essentials', 'Computers And Electric Accessories']
Length: 8, dtype: str


### Step 5: Parse Dates and Extract Time Features
**Why:** To perform time series analysis, we must convert `transaction_date` to datetime objects, and we will extract the month and year for monthly aggregation.

In [6]:
df['transaction_date'] = pd.to_datetime(df['transaction_date'])
df['transaction_year'] = df['transaction_date'].dt.year
df['transaction_month'] = df['transaction_date'].dt.month
df['transaction_day_of_week'] = df['transaction_date'].dt.day_name()
print("Date parsing successful.")

Date parsing successful.


### Step 6: Map Discount Applied to Boolean/Integer
**Why:** `discount_applied` might have mixed boolean strings ('True', 'False') or empty values. We will map this to a clean 0 or 1 integer format.

In [7]:
df['discount_applied'] = df['discount_applied'].replace({'True': 1, 'False': 0, True: 1, False: 0})
df['discount_applied'] = pd.to_numeric(df['discount_applied'], errors='coerce').fillna(0).astype(int)
print("Discount Applied mapped to 0/1 integers.")

Discount Applied mapped to 0/1 integers.


### Step 6b: Add Derived Features
**Why:** Generate additional metrics like revenue_bucket, avg_unit_price, and is_weekend for deeper dashboarding.

In [8]:
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from scripts.etl_pipeline import add_derived_features
df = add_derived_features(df)
print('Derived features added. New shape:', df.shape)

Derived features added. New shape: (11971, 19)


### Step 7: Export Cleaned Dataset
**Why:** Save the fully transformed dataset for EDA and statistical modeling.

In [9]:
PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_DATA_PATH, index=False)
print(f"Saved cleaned dataset to {PROCESSED_DATA_PATH}")
print(f"Final shape: {df.shape}")

Saved cleaned dataset to /Users/anshkumar/Desktop/DVA-Private/some/data/processed/cleaned_dataset.csv
Final shape: (11971, 19)
